# <font color = 'red'> Dependencias

In [136]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio, count_by_category, proportions_by_category, counts_and_proportions_by_category)
from visualization_tools import plot_interactive_chart, plot_categorical_proportions
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> Carga de Datos

In [137]:
col = "Payment_of_Min_Amount"

In [149]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [150]:
df = df.filter(regex = col + '|Credit_Mix|Credit_Score').drop('Binary_Credit_Score', axis = 1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Credit_Mix                 100000 non-null  object
 1   Payment_of_Min_Amount      100000 non-null  object
 2   Credit_Score               100000 non-null  int64 
 3   Payment_of_Min_Amount_No   100000 non-null  bool  
 4   Payment_of_Min_Amount_Yes  100000 non-null  bool  
dtypes: bool(2), int64(1), object(2)
memory usage: 2.5+ MB


In [151]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
Duplicate rows found: 99996


# <font color = 'red'> Análisis

## <font color = 'skyblue'> ANÁLISIS GENERAL

In [152]:
category_names = [i for i in df.drop(col, axis = 1).columns if i.startswith(col)]

results = counts_and_proportions_by_category(df, category_names, "Credit_Mix")
counts_df = results["counts"]
proportions_df = results["proportions"]
summary_df = results["summary"]

counts_df.index = counts_df.index.str.replace(col + '_', '')
proportions_df.index = proportions_df.index.str.replace(col + '_', '')
summary_df.index = summary_df.index.str.replace(col + '_', '')

In [153]:
custom_colors = {
    "prop_Bad": "orangered",
    "prop_Standard": "gold",
    "prop_Good": "skyblue"
}

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad", "prop_Standard", "prop_Good"],  
    stacked=False,                      
    title=f"Proporción de Credit Score por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=600,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()


fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad"],  
    stacked=False,                      
    title=f"Proporción de Bad por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Good"],  
    stacked=False,                      
    title=f"Proporción Good por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Good"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Standard"],  
    stacked=False,                      
    title=f"Proporción Standard por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Standard"          
)

fig.show()

# A simple vista sí parece que el impago y el pago del monto mínimo tienen relación:
#     * La muestra de deudores que no pagan el monto mínimo o son Good (75%) o Standard (25%) y no hay malos deudores.
#     * Por su parte, la muestra de deudores que sí pagan el monto mínimo o son Bad (40%) o Standard (60%) pero no hay malos.
# De modo que no hay deudores buenos que paguen el monto mínimo. Ni malos que no lo paguen. Los Standard, en cambio, están en las dos clases
# pero principalmente en los que pagan el monto mínimo.


<font color = 'skyblue'> Regresión

Los clientes que no pagan solo el monto mínimo tienen un Credit_Score promedio de 1.749, lo que equivale a una calificación cercana a Standard/Good.

Los clientes que sí pagan solo el monto mínimo tienen un Credit_Score promedio de 0.6001, lo que corresponde a una calificación entre Bad/Standard.

In [154]:
y_col = "Credit_Score"

df_ = df[[col]]

model = smf.ols(f"{y_col} ~ " + ' + '.join(df_.columns) + ' -1', data=df).fit() # f"{y_col} ~ " + ' + '.join(df_.columns)

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           Credit_Score   R-squared:                       0.592
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                 1.454e+05
Date:                Sun, 06 Apr 2025   Prob (F-statistic):               0.00
Time:                        12:49:40   Log-Likelihood:                -65937.
No. Observations:              100000   AIC:                         1.319e+05
Df Residuals:                   99998   BIC:                         1.319e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Payment_of_Min_Amoun

In [144]:
y_column = "Credit_Score"
base_category = "Payment_of_Min_Amount_No"

x_columns = [i for i in df.columns if col + '_' in i] # categories

x_columns.remove(base_category)

model = OrderedModel(df[y_column], df[x_columns], distr="logit")

result = model.fit(method='bfgs')

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.628553
         Iterations: 136
         Function evaluations: 166
         Gradient evaluations: 166
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -62855.
Model:                   OrderedModel   AIC:                         1.257e+05
Method:            Maximum Likelihood   BIC:                         1.257e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        12:44:42                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
--------------------

No pagar el monto mínimo se asocia con mejores categorías de crédito

Puntualmente, pasar de 0 a 1 en Payment_of_Min_Amount (de sí a no) incrementa la probabilidad de que Credit_Score esté en una categoría superior (de Bad a Standard, o de Standard a Good). 

Estar a la derecha de -0.4058 aumenta la probabilidad de pasar de Bad a Standard o Good.

Estar a la derecha de 2.9153 aumenta la probabilidad de pertenecer a la categoría Good.

In [ ]:
y_column = "Credit_Score"
x_column = col

df_ = df.copy()

mapping_dict = {
    "No": 1,
    "Yes": 0
}

df_[col] = df_[col].map(mapping_dict)

model = OrderedModel(df_[y_column], df_[x_column], distr="logit")
result = model.fit(method='bfgs')

# Mostrar resultados
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.628553
         Iterations: 160
         Function evaluations: 193
         Gradient evaluations: 193
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -62855.
Model:                   OrderedModel   AIC:                         1.257e+05
Method:            Maximum Likelihood   BIC:                         1.257e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        12:56:07                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
------------------------